# NB05: Segmented LightGBM Unit Demand Forecast Model

Three-segment architecture separating PC (very high volume), LO/ME (mid-volume seasonal), and the 8 low-volume intermittent divisions.
Revenue forecasting has been moved to NB07.

**Segment definitions:**
- `pc_only` — PC (chemicals): avg ~187 units/wk — regression/MAE
- `mid_vol` — LO, ME: avg 8–14 units/wk — tweedie, variance_power=1.2
- `low_vol` — BQ, CH, FI, GA, HT, PA, SP, TO: avg 1–5 units/wk — tweedie, variance_power=1.5


In [30]:
import sys
import os

# Dynamically resolve project root — works across sessions
# Fallback chain: environment variable > relative path > hardcoded
_this_dir = os.path.dirname(os.path.abspath('__file__'))
_candidates = [
    os.environ.get('UC4_PROJECT_ROOT', ''),
    os.path.join(_this_dir, '..'),  # if run from notebooks/
]
for _c in _candidates:
    _test = os.path.join(_c, 'data', 'processed', 'modeling_dataset.csv')
    if os.path.exists(_test):
        _project_root = _c
        break
else:
    raise FileNotFoundError("Cannot find project root. Set UC4_PROJECT_ROOT or run from notebooks/")

# Add .pylibs if it exists (for lightgbm, sklearn, etc.)
_pylibs = os.path.join(os.path.dirname(_project_root), '.pylibs')
if os.path.isdir(_pylibs):
    sys.path.insert(0, _pylibs)

import pandas as pd
import numpy as np
import lightgbm as lgb
from sklearn.metrics import r2_score, mean_absolute_error, mean_absolute_percentage_error
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# ========== PATHS ==========
project_root = Path(_project_root)
data_dir = project_root / 'data' / 'processed'
fig_dir = project_root / 'reports' / 'figures'
fig_dir.mkdir(parents=True, exist_ok=True)

# ========== 1. LOAD & PREPARE DATA ==========
print("=" * 60)
print("NB05: GLOBAL LIGHTGBM FORECAST MODEL")
print("=" * 60)

df = pd.read_csv(data_dir / 'modeling_dataset.csv')
df['week_ending'] = pd.to_datetime(df['week_ending'])
print(f"\nRaw data: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"Stores: {df['store_code'].nunique()}, Divisions: {df['division_code'].nunique()}")
print(f"Groups: {df.groupby(['store_code','division_code']).ngroups}")
print(f"Date range: {df['week_ending'].min().date()} to {df['week_ending'].max().date()}")

# ========== FEATURE DEFINITIONS ==========
TARGET = 'units'

DEMAND_FEATS = ['units_lag_1w', 'units_lag_2w', 'units_lag_4w', 'units_lag_52w',
                'units_roll4_mean', 'units_roll4_std', 'units_roll8_mean', 'units_roll8_std',
                'units_roll12_mean', 'units_roll12_std', 'units_volatility']
REVENUE_FEATS = ['revenue_lag_1w', 'revenue_lag_4w', 'revenue_roll4_mean',
                 'avg_price_per_unit', 'price_lag_1w']
SEASON_FEATS = ['sin_week_1', 'cos_week_1', 'sin_week_2', 'cos_week_2']
CALENDAR_FEATS = ['week_of_yr', 'month', 'year_idx', 'quarter',
                  'is_summer_peak', 'is_spring_opening', 'is_fall_closing', 'is_winter_off',
                  'n_holidays', 'has_holiday']
WEATHER_RAW = ['avg_temp', 'max_temp', 'total_precip', 'sunshine_hours', 'rain_days', 'snow_days', 'bad_weather_days']
WEATHER_DEV = ['avg_temp_dev', 'avg_temp_dev_z', 'total_precip_dev', 'total_precip_dev_z',
               'sunshine_hours_dev', 'sunshine_hours_dev_z', 'rain_days_dev', 'rain_days_dev_z',
               'snow_days_dev', 'snow_days_dev_z', 'bad_weather_days_dev', 'bad_weather_days_dev_z']
WEATHER_THRES = ['temp_above_20', 'temp_above_25', 'temp_below_0', 'cooling_degree_days', 'heating_degree_days']
WEATHER_LAG = ['avg_temp_lag_1w', 'avg_temp_lag_2w', 'temp_shock', 'temp_warming']
WEATHER_INTERACT = ['precip_x_summer', 'bad_wx_x_summer', 'sunshine_x_summer',
                    'temp_above_25_x_sum', 'precip_x_spring', 'temp_warming_x_spr']
ACTIVITY_FEATS = ['n_transactions', 'is_active_product', 'weeks_with_sales']

FEATURES = (DEMAND_FEATS + REVENUE_FEATS + SEASON_FEATS + CALENDAR_FEATS +
            WEATHER_RAW + WEATHER_DEV + WEATHER_THRES + WEATHER_LAG + WEATHER_INTERACT + ACTIVITY_FEATS)

# Verify all features exist
missing = [f for f in FEATURES if f not in df.columns]
if missing:
    print(f"WARNING: Missing features: {missing}")
    FEATURES = [f for f in FEATURES if f in df.columns]
print(f"\nFeatures: {len(FEATURES)} numeric")
# ========== THREE-SEGMENT VOLUME DEFINITION (Fix 1) ==========
# Root cause of LO/ME regression: they are mid-volume seasonal series (8-14 units/wk)
# — not smooth enough for regression/MAE (PC-only territory), but too dense for the
# sparse-count tweedie tuned for 1-5 unit/wk divisions. They need their own segment.
#
# Segment    Divisions        Avg units/wk  Objective
# pc_only    PC               ~187          regression (MAE)
# mid_vol    LO, ME           8–14          tweedie (vp=1.2, closer to Poisson)
# low_vol    BQ,CH,FI,GA,     1–5           tweedie (vp=1.5, more right-skewed)
#            HT,PA,SP,TO
PC_ONLY_DIVS = {'PC'}
MID_VOL_DIVS = {'LO', 'ME'}
LOW_VOL_DIVS = set(df['division_code'].unique()) - PC_ONLY_DIVS - MID_VOL_DIVS

def assign_segment(d):
    if d in PC_ONLY_DIVS:  return 'pc_only'
    if d in MID_VOL_DIVS:  return 'mid_vol'
    return 'low_vol'

df['vol_segment'] = df['division_code'].apply(assign_segment)
print(f"\nThree-segment split:")
for seg, divs in [('pc_only', PC_ONLY_DIVS), ('mid_vol', MID_VOL_DIVS), ('low_vol', LOW_VOL_DIVS)]:
    rows = (df['vol_segment'] == seg).sum()
    print(f"  {seg:<10}: {sorted(divs)}  — {rows:,} rows")


NB05: GLOBAL LIGHTGBM FORECAST MODEL

Raw data: 23,749 rows × 81 columns
Stores: 27, Divisions: 11
Groups: 287
Date range: 2023-11-05 to 2026-03-22

Features: 67 numeric

Three-segment split:
  pc_only   : ['PC']  — 2,995 rows
  mid_vol   : ['LO', 'ME']  — 5,578 rows
  low_vol   : ['BQ', 'CH', 'FI', 'GA', 'HT', 'PA', 'SP', 'TO']  — 15,176 rows


In [31]:
# ========== ENCODE CATEGORICALS ==========
from sklearn.preprocessing import LabelEncoder

le_store = LabelEncoder()
le_div = LabelEncoder()
df['store_enc'] = le_store.fit_transform(df['store_code'])
df['div_enc'] = le_div.fit_transform(df['division_code'])
CAT_ENC = ['store_enc', 'div_enc']

# ========== INTERMITTENCY FEATURES ==========
df = df.sort_values(['store_code', 'division_code', 'week_ending'])
for grp_cols in [['store_code', 'division_code']]:
    g = df.groupby(grp_cols)
    nonzero = (df['units'] > 0).astype(int)
    cumcount = nonzero.groupby([df['store_code'], df['division_code']]).cumsum()
    df['cumulative_sales_count'] = cumcount
    df['zero_frac_12w'] = 1 - g['units'].transform(lambda x: x.rolling(12, min_periods=4).apply(lambda w: (w>0).mean()))

INTERMIT_FEATS = ['cumulative_sales_count', 'zero_frac_12w']
FEATURES += INTERMIT_FEATS
print(f"Added intermittency features. Total: {len(FEATURES)} + {len(CAT_ENC)} categorical = {len(FEATURES)+len(CAT_ENC)}")

Added intermittency features. Total: 69 + 2 categorical = 71


In [32]:
# ========== DROP NaN WARMUP ==========
FEATURE_COLS = FEATURES + CAT_ENC
df_model = df.dropna(subset=['units_lag_4w', 'avg_temp_lag_2w']).copy()
print(f"After lag warmup drop: {len(df_model):,} rows "
      f"({df_model.groupby(['store_code','division_code']).ngroups} groups)")
print(f"Segment sizes after warmup drop:")
print(df_model.groupby('vol_segment')['division_code'].value_counts().to_string())


After lag warmup drop: 22,605 rows (284 groups)
Segment sizes after warmup drop:
vol_segment  division_code
low_vol      FI               2045
             GA               2031
             BQ               1958
             CH               1891
             SP               1791
             HT               1742
             TO               1477
             PA               1411
mid_vol      LO               2741
             ME               2629
pc_only      PC               2889


## 2. Walk-Forward Train/Test Split

In [33]:
# ========== 2. WALK-FORWARD SPLIT ==========
df_model = df_model.sort_values(['week_ending', 'store_code', 'division_code']).reset_index(drop=True)
all_weeks = sorted(df_model['week_ending'].unique())
n_weeks = len(all_weeks)
holdout_weeks = 26
cutoff_idx = n_weeks - holdout_weeks
cutoff_date = all_weeks[cutoff_idx]

train_mask = df_model['week_ending'] < cutoff_date
test_mask  = df_model['week_ending'] >= cutoff_date

# Per-segment boolean masks
pc_mask  = df_model['vol_segment'] == 'pc_only'
mid_mask = df_model['vol_segment'] == 'mid_vol'
low_mask = df_model['vol_segment'] == 'low_vol'

X_train = df_model.loc[train_mask, FEATURE_COLS]
y_train = df_model.loc[train_mask, TARGET]
X_test  = df_model.loc[test_mask,  FEATURE_COLS]
y_test  = df_model.loc[test_mask,  TARGET]

# PC-only segment
X_train_pc  = df_model.loc[train_mask & pc_mask,  FEATURE_COLS]
y_train_pc  = df_model.loc[train_mask & pc_mask,  TARGET]
X_test_pc   = df_model.loc[test_mask  & pc_mask,  FEATURE_COLS]
y_test_pc   = df_model.loc[test_mask  & pc_mask,  TARGET]

# Mid-volume segment
X_train_mid = df_model.loc[train_mask & mid_mask, FEATURE_COLS]
y_train_mid = df_model.loc[train_mask & mid_mask, TARGET]
X_test_mid  = df_model.loc[test_mask  & mid_mask, FEATURE_COLS]
y_test_mid  = df_model.loc[test_mask  & mid_mask, TARGET]

# Low-volume segment
X_train_low = df_model.loc[train_mask & low_mask, FEATURE_COLS]
y_train_low = df_model.loc[train_mask & low_mask, TARGET]
X_test_low  = df_model.loc[test_mask  & low_mask, FEATURE_COLS]
y_test_low  = df_model.loc[test_mask  & low_mask, TARGET]

print(f"Cutoff: {cutoff_date.date()}   holdout: {holdout_weeks} weeks")
print(f"Train — pc: {len(X_train_pc):,}  mid: {len(X_train_mid):,}  low: {len(X_train_low):,}  total: {len(X_train):,}")
print(f"Test  — pc: {len(X_test_pc):,}   mid: {len(X_test_mid):,}   low: {len(X_test_low):,}   total: {len(X_test):,}")
print(f"Features: {len(FEATURE_COLS)}")


Cutoff: 2025-09-28   holdout: 26 weeks
Train — pc: 2,223  mid: 4,230  low: 12,567  total: 19,020
Test  — pc: 666   mid: 1,140   low: 1,779   total: 3,585
Features: 71


## 3. LightGBM Point Forecast

In [34]:
# ========== 3. THREE-SEGMENT LIGHTGBM TRAINING (Fix 1) ==========
print("\n" + "=" * 60)
print("TRAINING THREE-SEGMENT LIGHTGBM MODELS")
print("=" * 60)

# ── PC-only params ────────────────────────────────────────────────────────────
# Smooth high-volume series — regression/MAE is appropriate.
# min_child_samples=15: compromise from Fix 2 (better than orig 20, safer than 10).
lgb_params_pc = {
    'objective': 'regression',
    'metric': 'mae',
    'boosting_type': 'gbdt',
    'n_estimators': 600,
    'max_depth': 6,
    'num_leaves': 31,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.5,
    'reg_lambda': 1.0,
    'min_child_samples': 15,
    'random_state': 42,
    'verbose': -1,
    'n_jobs': -1,
}

# ── Mid-volume params (LO, ME) ────────────────────────────────────────────────
# 8–14 units/wk with strong seasonality and off-season intermittency.
# tweedie variance_power=1.2: closer to Poisson — less skew than the sparse divisions.
# num_leaves=45: more capacity than low-vol (31) but less than what caused LO/ME overfitting.
# min_child_samples=10: finer splits justified by higher volume than low-vol.
lgb_params_mid = {
    'objective': 'tweedie',
    'tweedie_variance_power': 1.2,
    'metric': 'tweedie',
    'boosting_type': 'gbdt',
    'n_estimators': 600,
    'max_depth': 6,
    'num_leaves': 45,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 0.5,
    'reg_lambda': 1.0,
    'min_child_samples': 10,
    'random_state': 42,
    'verbose': -1,
    'n_jobs': -1,
}

# ── Low-volume params (BQ, CH, FI, GA, HT, PA, SP, TO) ───────────────────────
# Sparse intermittent series (1–5 units/wk). Unchanged from v1 — these params
# already delivered 40–72% wMAPE improvements across all 8 divisions.
lgb_params_low = {
    'objective': 'tweedie',
    'tweedie_variance_power': 1.5,
    'metric': 'tweedie',
    'boosting_type': 'gbdt',
    'n_estimators': 600,
    'max_depth': 5,
    'num_leaves': 31,
    'learning_rate': 0.05,
    'subsample': 0.8,
    'colsample_bytree': 0.8,
    'reg_alpha': 1.0,
    'reg_lambda': 2.0,
    'min_child_samples': 5,
    'random_state': 42,
    'verbose': -1,
    'n_jobs': -1,
}

# ── Train PC-only model ───────────────────────────────────────────────────────
print("\nTraining PC-ONLY model (PC)...")
model_pc = lgb.LGBMRegressor(**lgb_params_pc)
model_pc.fit(
    X_train_pc, y_train_pc,
    eval_set=[(X_test_pc, y_test_pc)],
    callbacks=[lgb.early_stopping(50, verbose=True), lgb.log_evaluation(100)]
)
y_pred_pc_train = np.maximum(model_pc.predict(X_train_pc), 0)
y_pred_pc_test  = np.maximum(model_pc.predict(X_test_pc),  0)
print(f"  Train MAE: {mean_absolute_error(y_train_pc, y_pred_pc_train):.2f}")
print(f"  Test  MAE: {mean_absolute_error(y_test_pc,  y_pred_pc_test):.2f}")
print(f"  Test  R²:  {r2_score(y_test_pc, y_pred_pc_test):.4f}")

# ── Train mid-volume model ────────────────────────────────────────────────────
print("\nTraining MID-VOLUME model (LO, ME)...")
model_mid = lgb.LGBMRegressor(**lgb_params_mid)
model_mid.fit(
    X_train_mid, y_train_mid,
    eval_set=[(X_test_mid, y_test_mid)],
    callbacks=[lgb.early_stopping(50, verbose=True), lgb.log_evaluation(100)]
)
y_pred_mid_train = np.maximum(model_mid.predict(X_train_mid), 0)
y_pred_mid_test  = np.maximum(model_mid.predict(X_test_mid),  0)
print(f"  Train MAE: {mean_absolute_error(y_train_mid, y_pred_mid_train):.2f}")
print(f"  Test  MAE: {mean_absolute_error(y_test_mid,  y_pred_mid_test):.2f}")
print(f"  Test  R²:  {r2_score(y_test_mid, y_pred_mid_test):.4f}")

# ── Train low-volume model ────────────────────────────────────────────────────
print("\nTraining LOW-VOLUME model (BQ, CH, FI, GA, HT, PA, SP, TO)...")
model_low = lgb.LGBMRegressor(**lgb_params_low)
model_low.fit(
    X_train_low, y_train_low,
    eval_set=[(X_test_low, y_test_low)],
    callbacks=[lgb.early_stopping(50, verbose=True), lgb.log_evaluation(100)]
)
y_pred_low_train = np.maximum(model_low.predict(X_train_low), 0)
y_pred_low_test  = np.maximum(model_low.predict(X_test_low),  0)
print(f"  Train MAE: {mean_absolute_error(y_train_low, y_pred_low_train):.2f}")
print(f"  Test  MAE: {mean_absolute_error(y_test_low,  y_pred_low_test):.2f}")
print(f"  Test  R²:  {r2_score(y_test_low, y_pred_low_test):.4f}")

# ── Stitch all three segments back into full row order ────────────────────────
y_pred_train = pd.Series(index=df_model.loc[train_mask].index, dtype=float)
y_pred_train.loc[X_train_pc.index]  = y_pred_pc_train
y_pred_train.loc[X_train_mid.index] = y_pred_mid_train
y_pred_train.loc[X_train_low.index] = y_pred_low_train

y_pred_test = pd.Series(index=df_model.loc[test_mask].index, dtype=float)
y_pred_test.loc[X_test_pc.index]  = y_pred_pc_test
y_pred_test.loc[X_test_mid.index] = y_pred_mid_test
y_pred_test.loc[X_test_low.index] = y_pred_low_test

y_pred_train = y_pred_train.values
y_pred_test  = y_pred_test.values

print(f"\nCombined test MAE: {mean_absolute_error(y_test, y_pred_test):.2f}")
print(f"Combined test R²:  {r2_score(y_test, y_pred_test):.4f}")

# lgb_params alias kept for any downstream reference
lgb_params = lgb_params_pc



TRAINING THREE-SEGMENT LIGHTGBM MODELS

Training PC-ONLY model (PC)...
Training until validation scores don't improve for 50 rounds
[100]	valid_0's l1: 30.0205
[200]	valid_0's l1: 28.2383
[300]	valid_0's l1: 28.1036
Early stopping, best iteration is:
[257]	valid_0's l1: 28.0004
  Train MAE: 38.36
  Test  MAE: 28.00
  Test  R²:  0.9546

Training MID-VOLUME model (LO, ME)...
Training until validation scores don't improve for 50 rounds
[100]	valid_0's tweedie: 50.7636
[200]	valid_0's tweedie: 50.6596
[300]	valid_0's tweedie: 50.625
[400]	valid_0's tweedie: 50.6159
Early stopping, best iteration is:
[441]	valid_0's tweedie: 50.6071
  Train MAE: 4.07
  Test  MAE: 7.31
  Test  R²:  0.3600

Training LOW-VOLUME model (BQ, CH, FI, GA, HT, PA, SP, TO)...
Training until validation scores don't improve for 50 rounds
[100]	valid_0's tweedie: 6.99997
[200]	valid_0's tweedie: 6.94321
Early stopping, best iteration is:
[212]	valid_0's tweedie: 6.93369
  Train MAE: 1.25
  Test  MAE: 1.37
  Test  R²:  

## 4. Quantile Regression — Confidence Intervals

In [35]:
# ========== 4. QUANTILE REGRESSION — CONFIDENCE INTERVALS (three segments) ==========
print("\n" + "=" * 60)
print("TRAINING QUANTILE MODELS (P5, P95) — three segments")
print("=" * 60)

# PC-only quantile
q_params_pc = {**lgb_params_pc, 'objective': 'quantile', 'metric': 'quantile'}
model_q05_pc = lgb.LGBMRegressor(**{**q_params_pc, 'alpha': 0.05})
model_q95_pc = lgb.LGBMRegressor(**{**q_params_pc, 'alpha': 0.95})
model_q05_pc.fit(X_train_pc, y_train_pc, eval_set=[(X_test_pc, y_test_pc)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
model_q95_pc.fit(X_train_pc, y_train_pc, eval_set=[(X_test_pc, y_test_pc)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])

# Mid-volume quantile (tweedie → quantile override)
q_params_mid = {**lgb_params_mid, 'objective': 'quantile', 'metric': 'quantile'}
model_q05_mid = lgb.LGBMRegressor(**{**q_params_mid, 'alpha': 0.05})
model_q95_mid = lgb.LGBMRegressor(**{**q_params_mid, 'alpha': 0.95})
model_q05_mid.fit(X_train_mid, y_train_mid, eval_set=[(X_test_mid, y_test_mid)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
model_q95_mid.fit(X_train_mid, y_train_mid, eval_set=[(X_test_mid, y_test_mid)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])

# Low-volume quantile (tweedie → quantile override)
q_params_low = {**lgb_params_low, 'objective': 'quantile', 'metric': 'quantile'}
model_q05_low = lgb.LGBMRegressor(**{**q_params_low, 'alpha': 0.05})
model_q95_low = lgb.LGBMRegressor(**{**q_params_low, 'alpha': 0.95})
model_q05_low.fit(X_train_low, y_train_low, eval_set=[(X_test_low, y_test_low)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])
model_q95_low.fit(X_train_low, y_train_low, eval_set=[(X_test_low, y_test_low)],
    callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(0)])

# Stitch CIs from all three segments
y_lower_s = pd.Series(index=df_model.loc[test_mask].index, dtype=float)
y_upper_s = pd.Series(index=df_model.loc[test_mask].index, dtype=float)
y_lower_s.loc[X_test_pc.index]  = np.maximum(model_q05_pc.predict(X_test_pc),   0)
y_upper_s.loc[X_test_pc.index]  = np.maximum(model_q95_pc.predict(X_test_pc),   0)
y_lower_s.loc[X_test_mid.index] = np.maximum(model_q05_mid.predict(X_test_mid), 0)
y_upper_s.loc[X_test_mid.index] = np.maximum(model_q95_mid.predict(X_test_mid), 0)
y_lower_s.loc[X_test_low.index] = np.maximum(model_q05_low.predict(X_test_low), 0)
y_upper_s.loc[X_test_low.index] = np.maximum(model_q95_low.predict(X_test_low), 0)

y_lower = np.minimum(y_lower_s.values, y_pred_test)
y_upper = np.maximum(y_upper_s.values, y_pred_test)

# Post-hoc CI widening (Fix 3) — keeps coverage near 90% target
CI_SCALE = 0.15
y_lower = np.maximum(y_pred_test - (y_pred_test - y_lower) * (1 + CI_SCALE), 0)
y_upper = y_pred_test + (y_upper - y_pred_test) * (1 + CI_SCALE)

coverage  = np.mean((y_test.values >= y_lower) & (y_test.values <= y_upper))
avg_width = np.mean(y_upper - y_lower)
print(f"\n90% Prediction Interval Coverage: {coverage:.1%} (target: 90%)")
print(f"Average interval width: {avg_width:.1f} units")



TRAINING QUANTILE MODELS (P5, P95) — three segments

90% Prediction Interval Coverage: 85.9% (target: 90%)
Average interval width: 37.3 units


In [36]:
# ========== BUILD TEST DATAFRAME ==========
def wmape(actual, predicted):
    denom = np.sum(np.abs(actual))
    return np.sum(np.abs(actual - predicted)) / denom if denom > 0 else np.nan

test_df = df_model.loc[test_mask].copy()
test_df['pred_units'] = y_pred_test
test_df['pred_units_lower'] = y_lower
test_df['pred_units_upper'] = y_upper
test_df['actual_units'] = y_test.values

overall_units_wmape = wmape(test_df['actual_units'].values, test_df['pred_units'].values)
print(f"\nOverall Units wMAPE: {overall_units_wmape:.4f}")


Overall Units wMAPE: 0.1892


## 5. Performance by Division & Store

In [37]:
# ========== 6. wMAPE HELPER + PER-DIVISION ==========
print("\n" + "=" * 60)
print("PERFORMANCE BY DIVISION")
print("=" * 60)

div_metrics = []
for div in sorted(test_df['division_code'].unique()):
    sub = test_df[test_df['division_code'] == div]
    if sub['actual_units'].sum() == 0:
        continue
    div_metrics.append({
        'division': div,
        'n_groups': sub.groupby('store_code').ngroups,
        'n_rows': len(sub),
        'avg_actual_units': round(sub['actual_units'].mean(), 1),
        'units_wmape': round(wmape(sub['actual_units'].values, sub['pred_units'].values), 3),
        'units_r2': round(r2_score(sub['actual_units'].values, sub['pred_units'].values), 3),
    })

div_df = pd.DataFrame(div_metrics).sort_values('avg_actual_units', ascending=False)
print(div_df.to_string(index=False))

print(f"\nOverall units wMAPE: {overall_units_wmape:.3f}")


PERFORMANCE BY DIVISION
division  n_groups  n_rows  avg_actual_units  units_wmape  units_r2
      PC        26     666             197.0        0.142     0.955
      ME        26     531              17.7        0.560     0.349
      LO        26     609              13.5        0.374     0.544
      GA        26     248               5.1        0.436     0.506
      TO        25     129               5.0        0.766    -0.006
      FI        25     459               4.1        0.263     0.762
      BQ        26     344               3.3        0.277     0.618
      CH        24     177               3.2        0.328     0.552
      PA        24     142               3.1        0.437     0.683
      SP        26     231               3.0        0.266     0.707
      HT        17      49               1.4        0.282     0.665

Overall units wMAPE: 0.189


In [38]:
# ========== 7. PER-STORE ==========
print("\n" + "=" * 60)
print("PERFORMANCE BY STORE (Top 15)")
print("=" * 60)

store_metrics = []
for store in sorted(test_df['store_code'].unique()):
    sub = test_df[test_df['store_code'] == store]
    if sub['actual_units'].sum() == 0:
        continue
    store_metrics.append({
        'store': store,
        'n_divs': sub['division_code'].nunique(),
        'units_wmape': round(wmape(sub['actual_units'].values, sub['pred_units'].values), 3),
    })

store_df = pd.DataFrame(store_metrics).sort_values('units_wmape')
print(store_df.head(15).to_string(index=False))

worst = store_df[store_df['units_wmape'] > 0.5]
if len(worst) > 0:
    print(f"\nWarning: {len(worst)} stores with wMAPE > 50%:")
    print(worst[['store', 'units_wmape']].to_string(index=False))


PERFORMANCE BY STORE (Top 15)
store  n_divs  units_wmape
 CP10      11        0.109
 CP02      10        0.123
 CP46       9        0.129
CP202      10        0.130
 CP04      10        0.132
 CP48      10        0.133
 CP40      10        0.145
 CP15      11        0.146
 CP05      11        0.148
 CP08      11        0.149
 CP45      10        0.152
 CP37      11        0.153
 CP35      11        0.154
 CP16      11        0.181
 CP32      10        0.183


## 6. Feature Importance

In [39]:
# ========== 8. FEATURE IMPORTANCE — three segments ==========
print("\n" + "=" * 60)
print("FEATURE IMPORTANCE (Top 15 per segment)")
print("=" * 60)

importance_pc = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': model_pc.feature_importances_,
    'segment': 'pc_only'
}).sort_values('importance', ascending=False)

importance_mid = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': model_mid.feature_importances_,
    'segment': 'mid_vol'
}).sort_values('importance', ascending=False)

importance_low = pd.DataFrame({
    'feature': FEATURE_COLS,
    'importance': model_low.feature_importances_,
    'segment': 'low_vol'
}).sort_values('importance', ascending=False)

importance_high = importance_pc.copy()  # backward compat alias
importance      = importance_pc.copy()  # backward compat alias

for label, imp in [('PC-only', importance_pc), ('Mid-vol (LO, ME)', importance_mid), ('Low-vol (8 divs)', importance_low)]:
    print(f"\n{label}:")
    print(imp.head(15)[['feature','importance']].to_string(index=False))

groups_map = {
    'Demand Lags': DEMAND_FEATS, 'Revenue/Price': REVENUE_FEATS,
    'Seasonality': SEASON_FEATS, 'Calendar': CALENDAR_FEATS,
    'Weather (raw)': WEATHER_RAW, 'Weather (derived)': WEATHER_DEV + WEATHER_THRES,
    'Weather (lags/interact)': WEATHER_LAG + WEATHER_INTERACT,
    'Identity': CAT_ENC, 'Activity': ACTIVITY_FEATS, 'Intermittency': INTERMIT_FEATS,
}
print("\nPC-only feature group contribution:")
total_imp = importance_pc['importance'].sum()
for name, feats in groups_map.items():
    present = [f for f in feats if f in importance_pc['feature'].values]
    grp_total = importance_pc.loc[importance_pc['feature'].isin(present), 'importance'].sum()
    print(f"  {name:25s}: {grp_total/total_imp*100:5.1f}%")



FEATURE IMPORTANCE (Top 15 per segment)

PC-only:
           feature  importance
    n_transactions         463
      units_lag_1w         297
avg_price_per_unit         273
sunshine_hours_dev         186
   units_roll4_std         182
    revenue_lag_1w         177
revenue_roll4_mean         165
         store_enc         141
     units_lag_52w         120
      units_lag_2w         119
  units_roll12_std         110
      temp_warming         102
   units_roll8_std          95
  units_roll4_mean          87
temp_warming_x_spr          86

Mid-vol (LO, ME):
           feature  importance
avg_price_per_unit        1399
    n_transactions        1312
      price_lag_1w         531
  units_volatility         465
         store_enc         437
  units_roll12_std         402
    revenue_lag_4w         397
   units_roll4_std         369
revenue_roll4_mean         368
      avg_temp_dev         359
      units_lag_1w         358
      units_lag_2w         354
  weeks_with_sales         350


## 7. Walk-Forward Cross-Validation

In [40]:
# ========== 9. WALK-FORWARD CV — three segments ==========
print("\n" + "=" * 60)
print("WALK-FORWARD CROSS-VALIDATION (6 folds, three segments)")
print("=" * 60)

HORIZON = 4
N_FOLDS = 6
MIN_TRAIN_WEEKS = 52
all_weeks_sorted = sorted(df_model['week_ending'].unique())
total_weeks = len(all_weeks_sorted)
fold_starts = np.linspace(MIN_TRAIN_WEEKS, total_weeks - HORIZON, N_FOLDS + 1, dtype=int)[:-1]

cv_results = []
for fold_i, start_idx in enumerate(fold_starts):
    test_start = all_weeks_sorted[start_idx]
    test_end   = all_weeks_sorted[min(start_idx + HORIZON - 1, total_weeks - 1)]

    tr = df_model['week_ending'] < test_start
    te = (df_model['week_ending'] >= test_start) & (df_model['week_ending'] <= test_end)
    if tr.sum() == 0 or te.sum() == 0:
        continue

    fold_pc  = lgb.LGBMRegressor(**{**lgb_params_pc,  'n_estimators': 300, 'verbose': -1})
    fold_mid = lgb.LGBMRegressor(**{**lgb_params_mid, 'n_estimators': 300, 'verbose': -1})
    fold_low = lgb.LGBMRegressor(**{**lgb_params_low, 'n_estimators': 300, 'verbose': -1})

    for fold_model, seg_mask in [(fold_pc, pc_mask), (fold_mid, mid_mask), (fold_low, low_mask)]:
        tr_seg = tr & seg_mask
        if tr_seg.sum() > 0:
            fold_model.fit(df_model.loc[tr_seg, FEATURE_COLS], df_model.loc[tr_seg, TARGET])

    y_hat_s = pd.Series(index=df_model.loc[te].index, dtype=float)
    for fold_model, seg_mask in [(fold_pc, pc_mask), (fold_mid, mid_mask), (fold_low, low_mask)]:
        te_seg = te & seg_mask
        if te_seg.sum() > 0:
            y_hat_s.loc[df_model.loc[te_seg].index] = np.maximum(
                fold_model.predict(df_model.loc[te_seg, FEATURE_COLS]), 0)

    y_hat  = y_hat_s.values
    y_true = df_model.loc[te, TARGET].values

    fold_mae   = mean_absolute_error(y_true, y_hat)
    fold_wmape = wmape(y_true, y_hat)
    fold_r2    = r2_score(y_true, y_hat)

    m = test_start.month
    season = 'Summer' if m in [6,7,8] else 'Winter' if m in [12,1,2] else 'Spring' if m in [3,4,5] else 'Fall'

    cv_results.append({
        'fold': fold_i + 1, 'train_rows': tr.sum(), 'test_rows': te.sum(),
        'test_period': f"{test_start.date()} to {test_end.date()}", 'season': season,
        'mae': fold_mae, 'wmape': fold_wmape, 'r2': fold_r2,
    })
    print(f"Fold {fold_i+1}: {test_start.date()} → {test_end.date()} ({season}) "
          f"| MAE={fold_mae:.2f}, wMAPE={fold_wmape:.3f}, R²={fold_r2:.3f}")

cv_df = pd.DataFrame(cv_results)
print(f"\nCV Mean wMAPE: {cv_df['wmape'].mean():.3f} ± {cv_df['wmape'].std():.3f}")
print(f"CV Mean MAE:   {cv_df['mae'].mean():.2f} ± {cv_df['mae'].std():.2f}")
print(f"CV Mean R²:    {cv_df['r2'].mean():.3f} ± {cv_df['r2'].std():.3f}")



WALK-FORWARD CROSS-VALIDATION (6 folds, three segments)
Fold 1: 2024-12-01 → 2024-12-22 (Winter) | MAE=5.14, wMAPE=0.203, R²=0.853
Fold 2: 2025-02-09 → 2025-03-02 (Winter) | MAE=5.70, wMAPE=0.303, R²=0.390
Fold 3: 2025-04-27 → 2025-05-18 (Spring) | MAE=37.13, wMAPE=0.161, R²=0.951
Fold 4: 2025-07-13 → 2025-08-03 (Summer) | MAE=12.94, wMAPE=0.093, R²=0.989
Fold 5: 2025-09-28 → 2025-10-19 (Fall) | MAE=12.73, wMAPE=0.159, R²=0.953
Fold 6: 2025-12-14 → 2026-01-04 (Winter) | MAE=6.43, wMAPE=0.231, R²=0.745

CV Mean wMAPE: 0.192 ± 0.072
CV Mean MAE:   13.35 ± 12.16
CV Mean R²:    0.813 ± 0.226


## 8. Baseline Comparison

In [41]:
# ========== 10. BASELINE COMPARISON ==========
print("\\n" + "=" * 60)
print("BASELINE COMPARISON")
print("=" * 60)

test_groups = df_model.loc[test_mask].copy()
test_groups['lgbm_pred'] = y_pred_test

test_groups['naive_seasonal'] = test_groups['units_lag_52w'].fillna(test_groups['units_lag_4w'])
test_groups['naive_ma4'] = test_groups['units_roll4_mean']
div_means = df_model.loc[train_mask].groupby('division_code')['units'].mean()
test_groups['naive_div_mean'] = test_groups['division_code'].map(div_means)

baselines = {
    'Seasonal Naive (52w)': 'naive_seasonal',
    'Moving Average (4w)': 'naive_ma4',
    'Division Mean': 'naive_div_mean',
    'Global LightGBM': 'lgbm_pred'
}

print(f"{'Model':<30s} {'MAE':>8s} {'wMAPE':>8s} {'R²':>8s}")
print("-" * 56)
for name, col in baselines.items():
    valid = test_groups[[TARGET, col]].dropna()
    if len(valid) == 0:
        continue
    mae = mean_absolute_error(valid[TARGET], valid[col])
    w = wmape(valid[TARGET].values, valid[col].values)
    r2 = r2_score(valid[TARGET], valid[col])
    marker = " <- OURS" if col == 'lgbm_pred' else ""
    print(f"{name:<30s} {mae:>8.2f} {w:>8.3f} {r2:>8.3f}{marker}")

\n============================================================
BASELINE COMPARISON
Model                               MAE    wMAPE       R²
--------------------------------------------------------
Seasonal Naive (52w)              20.27    0.467    0.747
Moving Average (4w)               20.96    0.483    0.767
Division Mean                    130.37    3.006   -3.108
Global LightGBM                    8.20    0.189    0.940 <- OURS


## 9. 4-Week Forward Forecast

In [42]:
# ========== 11. GENERATE 4-WEEK FORECAST (three segments) ==========
print("\n" + "=" * 60)
print("4-WEEK FORWARD FORECAST")
print("=" * 60)

latest_week = df_model['week_ending'].max()
latest_rows = (
    df_model.sort_values('week_ending')
    .groupby(['store_code', 'division_code'])
    .last()
    .reset_index()
)

SEG_MODELS = [
    ('pc_only',  model_pc,  model_q05_pc,  model_q95_pc),
    ('mid_vol',  model_mid, model_q05_mid, model_q95_mid),
    ('low_vol',  model_low, model_q05_low, model_q95_low),
]

forecast_rows = []
for horizon in range(1, 5):
    fwd = latest_rows.copy()
    fwd['forecast_week'] = latest_week + pd.Timedelta(weeks=horizon)
    fwd['horizon'] = horizon

    pred_s = pd.Series(index=fwd.index, dtype=float)
    lo_s   = pd.Series(index=fwd.index, dtype=float)
    hi_s   = pd.Series(index=fwd.index, dtype=float)

    for seg, m_pt, m_q05, m_q95 in SEG_MODELS:
        mask = fwd['vol_segment'] == seg
        if mask.sum() == 0:
            continue
        X_fwd_seg = fwd.loc[mask, FEATURE_COLS]
        pred_s.loc[mask] = np.maximum(m_pt.predict(X_fwd_seg),  0)
        lo_s.loc[mask]   = np.maximum(m_q05.predict(X_fwd_seg), 0)
        hi_s.loc[mask]   = np.maximum(m_q95.predict(X_fwd_seg), 0)

    fwd['pred_units']       = pred_s.values
    fwd['pred_units_lower'] = np.minimum(lo_s.values, pred_s.values)
    fwd['pred_units_upper'] = np.maximum(hi_s.values, pred_s.values)

    forecast_rows.append(fwd[[
        'store_code', 'division_code', 'forecast_week', 'horizon',
        'pred_units', 'pred_units_lower', 'pred_units_upper'
    ]])

forecast_df = pd.concat(forecast_rows, ignore_index=True)
print(f"Forecast rows: {len(forecast_df):,}  "
      f"({forecast_df['store_code'].nunique()} stores × "
      f"{forecast_df['division_code'].nunique()} divisions × 4 weeks)")
print(f"Total predicted units (4 weeks): {forecast_df['pred_units'].sum():,.0f}")
print("\nBy segment:")
for seg, divs in [('pc_only', PC_ONLY_DIVS), ('mid_vol', MID_VOL_DIVS), ('low_vol', LOW_VOL_DIVS)]:
    sub = forecast_df[forecast_df['division_code'].isin(divs)]
    print(f"  {seg}: {sorted(divs)} → {sub['pred_units'].sum():,.0f} units")



4-WEEK FORWARD FORECAST
Forecast rows: 1,136  (26 stores × 11 divisions × 4 weeks)
Total predicted units (4 weeks): 9,049

By segment:
  pc_only: ['PC'] → 5,913 units
  mid_vol: ['LO', 'ME'] → 1,644 units
  low_vol: ['BQ', 'CH', 'FI', 'GA', 'HT', 'PA', 'SP', 'TO'] → 1,493 units


## 10. Save Outputs

In [43]:
# ========== 12. SAVE OUTPUTS ==========
print("\n" + "=" * 60)
print("SAVING OUTPUTS")
print("=" * 60)

forecast_df.to_csv(data_dir / 'nb05_forecast_output.csv', index=False)
print("ok nb05_forecast_output.csv")

group_accuracy = []
for (s, d), g in test_df.groupby(['store_code', 'division_code']):
    group_accuracy.append({
        'store_code': s, 'division_code': d,
        'actual_units': g['actual_units'].sum(), 'pred_units': g['pred_units'].sum(),
        'units_wmape': wmape(g['actual_units'].values, g['pred_units'].values),
        'n_weeks': len(g),
    })
pd.DataFrame(group_accuracy).to_csv(data_dir / 'nb05_model_accuracy.csv', index=False)
print("ok nb05_model_accuracy.csv")

div_df.to_csv(data_dir / 'nb05_division_accuracy.csv', index=False)
print("ok nb05_division_accuracy.csv")

cv_df.to_csv(data_dir / 'nb05_cv_results.csv', index=False)
print("ok nb05_cv_results.csv")

importance_pc.to_csv(data_dir  / 'nb05_feature_importance_pc.csv',  index=False)
importance_mid.to_csv(data_dir / 'nb05_feature_importance_mid.csv', index=False)
importance_low.to_csv(data_dir / 'nb05_feature_importance_low.csv', index=False)
importance_high.to_csv(data_dir / 'nb05_feature_importance_high.csv', index=False)  # compat
importance.to_csv(data_dir     / 'nb05_feature_importance.csv',      index=False)   # compat
print("ok nb05_feature_importance_pc/mid/low.csv")

summary = {
    'model': 'Segmented LightGBM — 3 segments (pc_only regression | mid_vol tweedie vp1.2 | low_vol tweedie vp1.5)',
    'pc_only_divisions':  str(sorted(PC_ONLY_DIVS)),
    'mid_vol_divisions':  str(sorted(MID_VOL_DIVS)),
    'low_vol_divisions':  str(sorted(LOW_VOL_DIVS)),
    'train_rows': len(X_train), 'test_rows': len(X_test),
    'n_features': len(FEATURE_COLS),
    'n_groups': df_model.groupby(['store_code', 'division_code']).ngroups,
    'n_stores': df_model['store_code'].nunique(),
    'n_divisions': df_model['division_code'].nunique(),
    'test_units_wmape': overall_units_wmape,
    'test_units_r2': r2_score(y_test, y_pred_test),
    'cv_wmape_mean': cv_df['wmape'].mean(),
    'cv_wmape_std': cv_df['wmape'].std(),
    'units_pi_coverage_90': coverage,
}
pd.DataFrame([summary]).to_csv(data_dir / 'nb05_summary.csv', index=False)
print("ok nb05_summary.csv")

print("\n" + "=" * 60)
print("NB05 COMPLETE — FINAL SUMMARY")
print("=" * 60)
print(f"Architecture:  3-segment LightGBM (pc_only regression | mid_vol tweedie vp1.2 | low_vol tweedie vp1.5)")
print(f"Groups:        {df_model.groupby(['store_code','division_code']).ngroups}")
print(f"Train/Test:    {len(X_train):,} / {len(X_test):,}")
print(f"Features:      {len(FEATURE_COLS)}")
print(f"Units wMAPE:   {overall_units_wmape:.3f} (R²={r2_score(y_test, y_pred_test):.4f})")
print(f"CV wMAPE:      {cv_df['wmape'].mean():.3f} ± {cv_df['wmape'].std():.3f}")
print(f"90% CI:        {coverage:.1%}")



SAVING OUTPUTS
ok nb05_forecast_output.csv
ok nb05_model_accuracy.csv
ok nb05_division_accuracy.csv
ok nb05_cv_results.csv
ok nb05_feature_importance_pc/mid/low.csv
ok nb05_summary.csv

NB05 COMPLETE — FINAL SUMMARY
Architecture:  3-segment LightGBM (pc_only regression | mid_vol tweedie vp1.2 | low_vol tweedie vp1.5)
Groups:        284
Train/Test:    19,020 / 3,585
Features:      71
Units wMAPE:   0.189 (R²=0.9403)
CV wMAPE:      0.192 ± 0.072
90% CI:        85.9%
